#Imports

In [25]:
import pandas as pd
from nltk.lm import MLE
from nltk.util import ngrams
from nltk.lm import Vocabulary
import numpy as np
import re
from google.colab import drive

#Datasets loading, visualization and manipulation

**Trump DataFrame**

In [26]:
#https://github.com/MarkHershey/CompleteTrumpTweetsArchive (realDonaldTrump_in_office.csv)
#manually modified column "Tweet Text" by removing spaces and adding "_" in between the two strings
drive.mount('/content/gdrive',force_remount=True)
directory="/content/gdrive/MyDrive/Uni/aa2425/TLN/Radicioni/dataset/trump2.csv"

df_trump = pd.read_csv(directory,on_bad_lines="skip") #on_bad_lines="skip" since some records were wrongly formatted and generated error
print(f"Shape: {df_trump.shape}")
df_trump.head()

Mounted at /content/gdrive
Shape: (10684, 4)


,ID,Time,Tweet URL,Tweet_Text
0,@realDonaldTrump,2017-01-20 06:31,https://twitter.com/realDonaldTrump/status/82...,"""It all begins today! I will see you at 11:00..."
1,@realDonaldTrump,2017-01-20 11:54,https://twitter.com/realDonaldTrump/status/82...,"""We will bring back our jobs. We will bring b..."
2,@realDonaldTrump,2017-01-20 11:55,https://twitter.com/realDonaldTrump/status/82...,"""We will follow two simple rules: BUY AMERICA..."
3,@realDonaldTrump,2017-01-20 11:58,https://twitter.com/realDonaldTrump/status/82...,"""It is time to remember that...https://www.fa..."
4,@realDonaldTrump,2017-01-20 12:13,https://twitter.com/realDonaldTrump/status/82...,"""TO ALL AMERICANS https://www.facebook.com/Do..."


**Obama DataFrame**

In [27]:
#https://www.kaggle.com/datasets/neelgajare/all-12000-president-obama-tweets?resource=download
drive.mount('/content/gdrive',force_remount=True)
directory="/content/gdrive/MyDrive/Uni/aa2425/TLN/Radicioni/dataset/obama.csv"

df_obama = pd.read_csv(directory)
print(f"Shape: {df_obama.shape}")
df_obama.head()

Mounted at /content/gdrive
Shape: (11539, 11)


,UserScreenName,UserName,Timestamp,Text,Embedded_text,Emojis,Comments,Likes,Retweets,Image link,Tweet URL
0,Barack Obama,@BarackObama,2007-05-07T15:52:28.000Z,"Barack Obama\n@BarackObama\n·\nMay 7, 2007",At the Detroit Economic Club – Talking about t...,NaN,22,29,105,[],https://twitter.com/BarackObama/status/53427172
1,Barack Obama,@BarackObama,2007-05-08T21:01:32.000Z,"Barack Obama\n@BarackObama\n·\nMay 8, 2007",Thinking we can cut oil consumption by 2.5 mil...,NaN,27,14,36,[],https://twitter.com/BarackObama/status/55928192
2,Barack Obama,@BarackObama,2007-05-11T01:15:18.000Z,"Barack Obama\n@BarackObama\n·\nMay 10, 2007","In Indianola, Iowa and heading to Des Moines\n...",NaN,25,19,73,[],https://twitter.com/BarackObama/status/59358952
3,Barack Obama,@BarackObama,2007-05-16T19:47:37.000Z,"Barack Obama\n@BarackObama\n·\nMay 16, 2007",Thinking the President’s word is not the last ...,NaN,10,24,32,[],https://twitter.com/BarackObama/status/66208452
4,Barack Obama,@BarackObama,2007-05-14T17:25:24.000Z,"Barack Obama\n@BarackObama\n·\nMay 14, 2007","In Trenton, NJ at AFL-CIO Town Hall Meeting\n8...",NaN,89,60,179,[],https://twitter.com/BarackObama/status/63909652


**Keeping just the relevant columns**

In [28]:
df_trump = df_trump.drop(columns=df_trump.columns.difference(["Tweet_Text"]))
df_obama = df_obama.drop(columns=df_obama.columns.difference(["Embedded_text"]))

**Moby Dick as a single string**

In [29]:
drive.mount('/content/gdrive',force_remount=True)
directory="/content/gdrive/MyDrive/Uni/aa2425/TLN/Radicioni/dataset/moby-dick.txt"

moby_dick = ""
f = open(directory, "r")
for line in f:
  if line!=" " and line!="\n":
    clean_line = line.lower().strip() #remove the \n
    moby_dick = moby_dick + " " + clean_line
f.close()

#start of chapter 1. loomings
moby_dick = moby_dick[26103:]

Mounted at /content/gdrive


#Obama & Trump ngrams

##Tweets preprocessing

**Tweets URLs filtering, tokenization, padding and vocabulary creation**

In [30]:
def ngrams_sentences_preprocessing(corpus, n):
  sentences = []
  vocabulary = set()
  for sentence in corpus:
    #lowering
    sentence = str(sentence).lower()
    #tokenization
    URL_REGEX = re.compile(r"https?:\/\/\S+|www\.\S+", re.IGNORECASE) #remove urls

    #keep only relevant words, dates, numbers, hours
    TOKEN_REGEX = re.compile(r"\b(?:\d{1,2}:\d{2}|\d{1,2}[/-]\d{1,2}[/-]\d{2,4}|(?:a|p)\.m\.?|\d+(?:\.\d+)?|[a-zA-Z]+(?:['’][a-zA-Z]+)*(?:-[a-zA-Z]+(?:['’][a-zA-Z]+)*)*)\b",re.IGNORECASE)

    sentence = URL_REGEX.sub("", sentence)
    tokenized_sentence = TOKEN_REGEX.findall(sentence)
    #padding
    for i in range(n-1):
      tokenized_sentence.insert(0, "<s>")
      tokenized_sentence.append("</s>")
    #vocabulary
    for word in tokenized_sentence:
      vocabulary.add(word)
    sentences.append(tokenized_sentence)

  print(f"Tokenized and padded sentences:\n{sentences}")
  print(f"Vocabulary:\n{vocabulary}")
  return (sentences,[word for word in vocabulary])

##Bigrams

###Creation and visualization

**Dataframe conversion and bigrams creation**

In [31]:
trump_corpus = df_trump.to_numpy()
obama_corpus = df_obama.to_numpy()

#List[List[Tuple]], lista delle liste degli ngrammi
trump_sentences,trump_vocabulary = ngrams_sentences_preprocessing(trump_corpus, 2)
trump_bigrams = [list(ngrams(sentence,n=2)) for sentence in trump_sentences]

#List[List[Tuple]], lista delle liste degli ngrammi
obama_sentences,obama_vocabulary = ngrams_sentences_preprocessing(obama_corpus, 2)
obama_bigrams = [list(ngrams(sentence,n=2)) for sentence in obama_sentences]

Output hidden; open in https://colab.research.google.com to view.

**Bigrams visualization**

In [32]:
# for phrases in obama_bigrams:
#   for tuples in phrases:
#     print(tuples)

###Models training

**MLE model fitting to bigrams and vocabulary**

In [33]:
bigram_trump_model = MLE(2)
bigram_trump_model.fit(trump_bigrams, vocabulary_text=Vocabulary(trump_vocabulary))

bigram_obama_model = MLE(2)
bigram_obama_model.fit(obama_bigrams, vocabulary_text=Vocabulary(obama_vocabulary))

###Text generation

In [34]:
def generate_sent(model, random_seed, num_words, text_seed):
  cond = False
  while(not cond):
    content = []
    try:
      for token in model.generate(num_words, random_seed=random_seed, text_seed=text_seed):
        if token == '<s>':
          continue
        if token == '</s>':
          cond = True
          break
        content.append(token)
    except:
      ...
  return content

**Trump ones**

In [54]:
for i in range(5):
  print(f"{" ".join(generate_sent(bigram_trump_model, random_seed=None, num_words=15, text_seed=["<s>"]))}\n")

why the american seniors will be in the new jobs justices or not done

thank you for the corner this sad and the new high crimes where we

a serious evidence that presents to frontline covid patients who will say potus realdonaldtrump

rt flotus wonderful woman walking hundreds of the president thank you nfib our history

the people never supporting military must get out of anarchists aren’t impeaching her way



**Obama ones**

In [50]:
for i in range(5):
  print(f"\n{" ".join(generate_sent(bigram_obama_model, random_seed=None, num_words=15, text_seed=["<s>"]))}")


the phone and waters it tells his progressive organizing for you we're kicking off

in new number of the affordable care about the housing crisis degreesnotdebt 773 228

watch last night is expensive wait president obama barackobama com grader gives another 073

rt n obamacare is in the deficit here's to families president obama this done

tell congress could ever wonder what this right one thing that health coverage progress


##Trigrams

###Creation and visualization

**Trigrams creation**

In [37]:
#List[List[Tuple]], lista delle liste degli ngrammi
trump_sentences,trump_vocabulary = ngrams_sentences_preprocessing(trump_corpus, 3)
trump_trigrams = [list(ngrams(sentence,n=3)) for sentence in trump_sentences]

#List[List[Tuple]], lista delle liste degli ngrammi
obama_sentences,obama_vocabulary = ngrams_sentences_preprocessing(obama_corpus, 3)
obama_trigrams = [list(ngrams(sentence,n=3)) for sentence in obama_sentences]

Output hidden; open in https://colab.research.google.com to view.

**Trigrams visualization**

In [38]:
# print("Obama trigrams")
# print trigrams
# for phrases in obama_trigrams:
#   for tuples in phrases:
#     print(tuples)

###Model training

In [39]:
trigram_trump_model = MLE(3)
trigram_trump_model.fit(trump_trigrams, vocabulary_text=Vocabulary(trump_vocabulary))

trigram_obama_model = MLE(3)
trigram_obama_model.fit(obama_trigrams, vocabulary_text=Vocabulary(obama_vocabulary))

###Text generation

**Trump ones**

In [52]:
for i in range(5):
  print(f"\n{" ".join(generate_sent(trigram_trump_model, random_seed=None, num_words=20, text_seed=["<s>","<s>"]))}")


thank you and now dems in despair republicans united realdonaldtrump survives amp impeachment all over the last 4 years

the jexodus movement encourages jewish people to see how many people both inside and outside of government ingrahamangle

rt tomfitton coup update bruce ohr wrote christopher steele admits using posts by random individuals from cnn to fraudnewscnn

i’ve done more in the kate steinle case no wonder their news conference live from the election fairly

maduro willing to negotiate democrats did nothing but more misery end the democrats impeachment stunt arguments with scalpel-like precision


**Obama ones**

In [41]:
for i in range(5):
  print(f"{" ".join(generate_sent(trigram_obama_model, random_seed=None, num_words=20, text_seed=["<s>","<s>"]))}\n")

instead of outsourcing are thinking of the disability rights activists who have shared their accounts of inauguration day

happening now president obama delivers the commencement address at barnard college in new york 13 year high actonjobs 065

when the dust has settled there can be yours but only when we can’t afford a tax hike 345

think the candidates as they know students can’t afford to double interest rates on federal lands a step further

all this takes eliminating the filibuster a jim crow relic then that’s what keeps us strong president obama



#Moby Dick ngrams

##Moby Dick text preprocessing

**ngrams and vocabulary generation**

In [42]:
def moby_dick_ngrams_sentences_preprocessing(corpus, n):
  sentences = []
  vocabulary = set()

  #Removing grammar characters, except ! ? . '
  moby_dick = corpus.replace("!"," !").replace("?"," ?").replace(",","").replace(":","").replace(";","").replace("”","").replace("“","").replace("_","").replace("-"," ").replace("—","").replace("  "," ").replace("*","")
  moby_dick = moby_dick.split(".")

  #sentences stripping
  for i in range(len(moby_dick)):
    moby_dick[i] = moby_dick[i].strip()

  for sentence in moby_dick:
    #tokenization
    tokenized_sentence = sentence.split(" ")
    #padding
    for i in range(n-1):
      tokenized_sentence.insert(0, "<s>")
      tokenized_sentence.append("</s>")
    #vocabulary
    for word in tokenized_sentence:
      vocabulary.add(word)
    sentences.append(tokenized_sentence)

  print(f"Tokenized and padded sentences:\n{sentences}")
  print(f"Vocabulary:\n{vocabulary}")
  return (sentences,[word for word in vocabulary])

##Bigrams

###Creation and visualization

In [43]:
moby_dick_sentences,moby_dick_vocabulary = moby_dick_ngrams_sentences_preprocessing(moby_dick, 2)
moby_dick_bigrams = [list(ngrams(sentence,n=2)) for sentence in moby_dick_sentences]

Tokenized and padded sentences:
[['<s>', 'chapter', '1', '</s>'], ['<s>', 'loomings', '</s>'], ['<s>', 'call', 'me', 'ishmael', '</s>'], ['<s>', 'some', 'years', 'agonever', 'mind', 'how', 'long', 'preciselyhaving', 'little', 'or', 'no', 'money', 'in', 'my', 'purse', 'and', 'nothing', 'particular', 'to', 'interest', 'me', 'on', 'shore', 'i', 'thought', 'i', 'would', 'sail', 'about', 'a', 'little', 'and', 'see', 'the', 'watery', 'part', 'of', 'the', 'world', '</s>'], ['<s>', 'it', 'is', 'a', 'way', 'i', 'have', 'of', 'driving', 'off', 'the', 'spleen', 'and', 'regulating', 'the', 'circulation', '</s>'], ['<s>', 'whenever', 'i', 'find', 'myself', 'growing', 'grim', 'about', 'the', 'mouth', 'whenever', 'it', 'is', 'a', 'damp', 'drizzly', 'november', 'in', 'my', 'soul', 'whenever', 'i', 'find', 'myself', 'involuntarily', 'pausing', 'before', 'coffin', 'warehouses', 'and', 'bringing', 'up', 'the', 'rear', 'of', 'every', 'funeral', 'i', 'meet', 'and', 'especially', 'whenever', 'my', 'hypos', 

###Model training and text generation

In [44]:
bigram_moby_dick_model = MLE(2)
bigram_moby_dick_model.fit(moby_dick_bigrams, vocabulary_text=Vocabulary(moby_dick_vocabulary))

In [45]:
for i in range(5):
  print(f"{" ".join(generate_sent(bigram_moby_dick_model, random_seed=None, num_words=15, text_seed=["<s>"]))}\n")

captain was near the death throbs of his crew pull on his final perch

all right have been gradually the two whales is far the living people’s noses

but this difference that this once comprehending it up the pepper and now if

but still maintained their calm enticing calm things to it many white mist it

hast seen old mast so that missing starbuck thou ?dost not grasp leaving mrs



##Trigrams

###Creation and visualization

In [46]:
moby_dick_sentences,moby_dick_vocabulary = moby_dick_ngrams_sentences_preprocessing(moby_dick, 3)
moby_dick_trigrams = [list(ngrams(sentence,n=3)) for sentence in moby_dick_sentences]

Tokenized and padded sentences:
[['<s>', '<s>', 'chapter', '1', '</s>', '</s>'], ['<s>', '<s>', 'loomings', '</s>', '</s>'], ['<s>', '<s>', 'call', 'me', 'ishmael', '</s>', '</s>'], ['<s>', '<s>', 'some', 'years', 'agonever', 'mind', 'how', 'long', 'preciselyhaving', 'little', 'or', 'no', 'money', 'in', 'my', 'purse', 'and', 'nothing', 'particular', 'to', 'interest', 'me', 'on', 'shore', 'i', 'thought', 'i', 'would', 'sail', 'about', 'a', 'little', 'and', 'see', 'the', 'watery', 'part', 'of', 'the', 'world', '</s>', '</s>'], ['<s>', '<s>', 'it', 'is', 'a', 'way', 'i', 'have', 'of', 'driving', 'off', 'the', 'spleen', 'and', 'regulating', 'the', 'circulation', '</s>', '</s>'], ['<s>', '<s>', 'whenever', 'i', 'find', 'myself', 'growing', 'grim', 'about', 'the', 'mouth', 'whenever', 'it', 'is', 'a', 'damp', 'drizzly', 'november', 'in', 'my', 'soul', 'whenever', 'i', 'find', 'myself', 'involuntarily', 'pausing', 'before', 'coffin', 'warehouses', 'and', 'bringing', 'up', 'the', 'rear', 'of',

###Model training and text generation

In [47]:
trigram_moby_dick_model = MLE(3)
trigram_moby_dick_model.fit(moby_dick_trigrams, vocabulary_text=Vocabulary(moby_dick_vocabulary))

In [49]:
for i in range(5):
  print(f"{" ".join(generate_sent(trigram_moby_dick_model, random_seed=None, num_words=15, text_seed=["<s>","<s>"]))}\n")

so next morning early leaving queequeg shut up in the course of the tackle

he is like the true method of absorbing it into thy hands starbuck

ever since those inventive but unscrupulous times when the swift sudden turn of death

this boat on yonder island and he heaved it up in some similar manner

let us now note what is it at all events the whole case

